# BudgiBrain
## Built with LangGraph, Semantic Memory, and Episodic Memory

### Dependencies and Requirements

In [ ]:
# Insalling pip dependecies
%pip install -r requirements.txt

In [46]:
# Importing all packages
from langgraph.graph import StateGraph, END
from langchain.docstore.document import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from pydantic import BaseModel, Field
from typing import Optional, List
from datetime import datetime
from IPython.display import display, clear_output
from dotenv import load_dotenv
import ipywidgets as widgets
from uuid import uuid4
import os
import re

# Loading env variables
load_dotenv()

True

In [ ]:
# Global variables
transaction_db = []

### Setting up Vector DB for Sematic Memory

In [ ]:
# Some common tools that will be leveraged frequently

# Splitting of sample data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)


# Setting up embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [48]:
# Sample training data (labelled user examples)
training_examples = [
    {
        "input": "Paid rent for August",
        "amount": None,
        "item_name": "rent",
        "category": "Rent"
    },
    {
        "input": "Gave landlord 12,000 for rent",
        "amount": 12000,
        "item_name": "rent",
        "category": "Rent"
    },
    {
        "input": "Netflix charged my card",
        "amount": None,
        "item_name": "Netflix",
        "category": "Subscriptions"
    },
    {
        "input": "Monthly subscription to Netflix",
        "amount": None,
        "item_name": "Netflix",
        "category": "Subscriptions"
    },
    {
        "input": "Bought veggies and snacks from DMart",
        "amount": None,
        "item_name": "DMart",
        "category": "Groceries"
    },
    {
        "input": "Shopping at Walmart – groceries and drinks",
        "amount": None,
        "item_name": "Walmart",
        "category": "Groceries"
    },
    {
        "input": "Recharge for electricity board",
        "amount": None,
        "item_name": "electricity board",
        "category": "Utilities"
    },
    {
        "input": "TNEB current bill paid",
        "amount": None,
        "item_name": "TNEB",
        "category": "Utilities"
    },
    {
        "input": "Watched Barbie movie at INOX",
        "amount": None,
        "item_name": "INOX",
        "category": "Shopping & Entertainment"
    },
    {
        "input": "Cinema with friends at PVR",
        "amount": None,
        "item_name": "PVR",
        "category": "Shopping & Entertainment"
    },
    {
        "input": "Renewed Tata AIG insurance for car",
        "amount": None,
        "item_name": "Tata AIG",
        "category": "Insurance"
    },
    {
        "input": "Paid ICICI car insurance",
        "amount": None,
        "item_name": "ICICI",
        "category": "Insurance"
    },
    {
        "input": "Consultation at Apollo Hospital",
        "amount": None,
        "item_name": "Apollo Hospital",
        "category": "Health"
    },
    {
        "input": "Doctor visit charges",
        "amount": None,
        "item_name": "Doctor",
        "category": "Health"
    },
    {
        "input": "Recharged metro card",
        "amount": None,
        "item_name": "metro card",
        "category": "Transport"
    },
    {
        "input": "Added ₹200 to metro pass",
        "amount": 200,
        "item_name": "metro pass",
        "category": "Transport"
    },
]

# Convert sample data to documents with category label as metadata
docs = []
for example in training_examples:
    docs.append(Document(
        page_content=example["input"],
        metadata={
            "doc_id": str(uuid4()),
            "input": example["input"],
            "amount": example["amount"],
            "item_name": example["item_name"],
            "category": example["category"],
            "action": "add",
            "source": "training_data"
        }
    ))

split_docs = text_splitter.split_documents(docs)

In [49]:
# Load or create FAISS vector store
if os.path.exists("faiss_store/index.faiss"):
    db = FAISS.load_local(
        "faiss_store",
        embeddings=embedding_model,
        allow_dangerous_deserialization=True
    )
    print("Using existing FAISS vector DB")
else:
    db = FAISS.from_documents(docs, embedding_model)
    db.save_local("faiss_store")
    print("Created and saved new FAISS vector DB from training examples")

Using existing FAISS vector DB


### Setting up LLM

In [50]:
# Initialize LLM
LLM = ChatGroq(
    model_name=os.environ.get("LITELLM_MODEL"),
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

### API Tools used by the LLM

In [ ]:
# API Tools that will be used by the LLM
@tool
def add_transaction(amount: Optional[float], category: str, item_name: str, input: str) -> dict:
    """Add a new transaction to the transaction_db."""
    
    transaction = {
        "datetime": datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        "amount": amount,
        "category": category,
        "item_name": item_name,
        "input": input
    }
    
    # Save to in-memory DB
    transaction_db.append(transaction)

    # Save to vector DB
    doc_id = str(uuid4())
    doc = Document(
        page_content=input,
        metadata={
            "doc_id": doc_id,
            "amount": amount,
            "category": category,
            "item_name": item_name,
            "datetime": transaction["datetime"],
            "source": "user"
        }
    )
    db.add_documents([doc])
    db.save_local("faiss_store")

    print("✅ Stored transaction in FAISS vector DB")
    return transaction

@tool
def edit_transaction(
    input: str,
    amount: Optional[float] = None,
    category: Optional[str] = None,
    item_name: Optional[str] = None
) -> dict:
    """
    Edit a transaction by searching for the most similar past transaction using the user input.
    Can update amount, category, and/or item_name if provided.
    """
    # Retrieve similar transactions from vector DB
    similar_docs = db.similarity_search(input, k=1)
    if not similar_docs:
        return {"error": "No matching transaction found to edit."}
    
    best_match = similar_docs[0]
    metadata = best_match.metadata
    original_input = best_match.page_content

    # Locate and update the transaction in memory
    for i, t in enumerate(transaction_db):
        if (
            t["input"] == original_input
            and t["datetime"] == metadata.get("datetime")
        ):
            updated_transaction = t.copy()
            
            # Update fields if new values are provided
            if amount is not None:
                updated_transaction["amount"] = amount
            if category is not None:
                updated_transaction["category"] = category
            if item_name is not None:
                updated_transaction["item_name"] = item_name

            # Update both in-memory DB and vector DB
            transaction_db[i] = updated_transaction
            
            # Delete old doc from FAISS using doc_id (if available)
            doc_id = metadata.get("doc_id")
            if doc_id:
                db.delete([doc_id])
            else:
                print("⚠️ Warning: doc_id missing; skipping delete")

            # Add updated doc with new doc_id
            new_doc_id = str(uuid4())
            new_doc = Document(
                page_content=input,
                metadata={
                    "doc_id": new_doc_id,
                    "datetime": updated_transaction["datetime"],
                    "amount": updated_transaction["amount"],
                    "category": updated_transaction["category"],
                    "item_name": updated_transaction["item_name"],
                    "input": input,
                    "source": "user"
                }
            )
            db.add_documents([new_doc])
            db.save_local("faiss_store")  # Persist changes
            
            return {
                "updated_transaction": updated_transaction,
                "matched_on": original_input
            }

    return {"error": "Transaction found in vector DB but not in in-memory DB."}

@tool
def search_transaction_by_category(category: str) -> List[dict]:
    """Search for transactions by category."""
    results = []
    for t in transaction_db:
        if (t['category'] == category):
            results.append(t)
    return results


@tool
def get_recent_similar_transactions(input: str, k: int = 3) -> List[dict]:
    """
    Get recent transactions similar to the user input from the vector DB.
    Returns the top-k most recent similar transactions.
    """
    similar_docs = db.similarity_search(input, k=10, filter={"source": "user"})  # get more for filtering

    if not similar_docs:
        return []

    # Filter and sort by datetime (descending)
    parsed_docs = []
    for doc in similar_docs:
        metadata = doc.metadata
        dt_str = metadata.get("datetime")
        try:
            dt = datetime.strptime(dt_str, '%Y-%m-%d %H:%M:%S')
        except Exception:
            dt = datetime.min  # fallback for malformed datetimes

        parsed_docs.append({
            "input": doc.page_content,
            "amount": metadata.get("amount"),
            "category": metadata.get("category"),
            "item_name": metadata.get("item_name"),
            "datetime": dt_str,
            "datetime_obj": dt
        })

    # Sort by most recent
    parsed_docs.sort(key=lambda x: x["datetime_obj"], reverse=True)

    # Return top-k without the datetime_obj helper
    return [{k: v for k, v in doc.items() if k != "datetime_obj"} for doc in parsed_docs[:k]]


tools = [add_transaction, edit_transaction, search_transaction_by_category, get_recent_similar_transactions]

### Building actual Graph

In [ ]:
# State Object that is passed in Graph
class TransactionState(BaseModel):
    input: str = ""
    amount: Optional[float] = None
    category: Optional[str] = None
    item_name: Optional[str] = None
    action: Optional[str] = None
    result: Optional[dict] = None
    parse_attempts: int = 0

# Output Parser 
class TransactionParse(BaseModel):
    action: str
    amount: Optional[float]
    category: Optional[str]
    item_name: Optional[str]

def retrieve_similar_examples(query: str, k: int = 3) -> List[Document]:
    return db.similarity_search(query, k=k)

# Node function of Graph
def llm_parse_node(state: TransactionState) -> TransactionState:
    print("➡️ Running llm_parse_node")
    state.parse_attempts += 1

    # Fetch similar examples
    similar_docs = retrieve_similar_examples(state.input)
    context = "\n".join(
        f"- {doc.page_content} ({doc.metadata.get('category')})"
        for doc in similar_docs
    )

    print("🧠 Similar examples:\n", context)

    # Compose context-enhanced prompt
    prompt_template = PromptTemplate.from_template(
        """
        You are a transaction assistant. Use the examples below to help understand the user's input.

        Examples:
        {context}

        Now extract the following fields from the user input:
        - action: one of "add", "edit", "search_by_category", "get_recent"
        - amount: the transaction amount (float or null)
        - category: the transaction category mentioned below
        - item_name: the item name (string or null)

        Categories:
        - Rent
        - Insurance
        - Utilities
        - Shopping & Entertainment
        - Groceries
        - Subscriptions
        - Transport
        - Health

        If a field is missing, set it to null.

        User input: {input}

        {format_instructions}
        """
    )

    parser = PydanticOutputParser(pydantic_object=TransactionParse)
    format_instructions = parser.get_format_instructions()
    chain = prompt_template | LLM | parser
    parsed: TransactionParse = chain.invoke({"input": state.input, "context": context, "format_instructions": format_instructions})
    

    print("🧠 Parsed Output:", parsed)

    state.action = parsed.action
    state.amount = parsed.amount
    state.category = parsed.category
    state.item_name = parsed.item_name

    return state

def add_node(state: TransactionState) -> TransactionState:
    result = add_transaction.invoke({
				"amount": state.amount,
				"category": state.category,
				"item_name": state.item_name,
				"input": state.input
		})
    state.result = result
    return state


def edit_node(state: TransactionState) -> TransactionState:
    result = edit_transaction.invoke({
        "input": state.input,
				"category": state.category,
				"item_name": state.item_name,
				"amount": state.amount
		})
    state.result = result
    return state

def search_node_by_category(state: TransactionState) -> TransactionState:
    result = search_transaction_by_category.invoke({
				"category": state.category
		})
    state.result = result
    return state

def get_recent_node(state: TransactionState) -> TransactionState:
    result = get_recent_similar_transactions.invoke({
        "input": state.input,
        "k": 1  
    })
    state.result = result
    return state

# Decision Function
def decide_next(state: TransactionState):
    print("🔄 Deciding next step...")
    print("State:", state)

    max_attempts = 2

     # Stop and return error if attempts exceeded
    if state.parse_attempts >= max_attempts:
        state.result = {
            "error": "❌ Unable to understand input after multiple attempts. Please rephrase."
        }
        return END

    if state.action == "add" and (state.amount is None or state.category is None or state.item_name is None):
        print("🔁 Re-entering llm_parse (missing add fields)")
        return "llm_parse"
    if state.action == "edit" and (state.category is None or state.item_name is None):
        print("🔁 Re-entering llm_parse (missing edit fields)")
        return "llm_parse"
    if state.action == "add":
        print("✅ Going to add")
        return "add"
    if state.action == "edit":
        print("✅ Going to edit")
        return "edit"
    if state.action == "search_by_category":
        print("✅ Going to search by category")
        return "search_by_category"
    if state.action == "get_recent":
      print("✅ Going to get_recent")
      return "get_recent"
    print("⏹ Ending graph")
    return END

# Build the Graph
def build_transaction_graph():
    graph = StateGraph(TransactionState)
    graph.add_node("llm_parse", llm_parse_node)
    graph.add_node("add", add_node)
    graph.add_node("edit", edit_node)
    graph.add_node("search_by_category", search_node_by_category)
    graph.add_node("get_recent", get_recent_node)
    graph.add_edge("get_recent", END)
    graph.add_edge("add", END)
    graph.add_edge("edit", END)
    graph.add_edge("search_by_category", END)
    graph.add_conditional_edges("llm_parse", decide_next)
    graph.set_entry_point("llm_parse")
    return graph.compile()

transaction_graph = build_transaction_graph()

# Example to call
def call_transaction_agent(user_input: str):
    print(f"🔍 User Input: {user_input}\n")
    state = TransactionState(input=user_input)
    final_state = transaction_graph.invoke(state)
    result = final_state["result"]

    # Display result
    print("📦 Parsed Result:")
    if isinstance(result, dict):
        for key, value in result.items():
            print(f"  - {key}: {value}")
    else:
        print(result)

    return result

In [ ]:
# Interactive_chat
def interactive_chat():
    input_box = widgets.Text(
        description='Prompt:',
        placeholder='e.g. Add 500 for groceries as milk',
        layout=widgets.Layout(width='90%')
    )
    output_area = widgets.Output()

    def on_enter(_):
        user_input = input_box.value.strip()
        if user_input:
            result = call_transaction_agent(user_input)
            with output_area:
                clear_output(wait=True)
                print("Result:", result)
            input_box.value = ''

    input_box.on_submit(on_enter)
    display(input_box, output_area)

In [54]:
call_transaction_agent("spent 111 on an aeroplane")

🔍 User Input: bought a car

➡️ Running llm_parse_node
🧠 Similar examples:
 - spent 5000 on a car (Transport)
- Bought a cow for 2 (Shopping & Entertainment)
- bought a cow for 1 (Shopping & Entertainment)


/opt/homebrew/Caskroom/miniconda/base/envs/budgibot-backend/lib/python3.11/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


InternalServerError: Error code: 500 - {'error': {'message': 'Internal Server Error', 'type': 'internal_server_error'}}

In [41]:
interactive_chat()

/var/folders/60/jkpcvndn6j1bchnq17ptwvm80000gn/T/ipykernel_56758/816961818.py:19: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  input_box.on_submit(on_enter)


Text(value='', description='Prompt:', layout=Layout(width='90%'), placeholder='e.g. Add 500 for groceries as m…

Output()

### Utility Blocks

In [ ]:
# Print size of locally stored vector db
db = FAISS.load_local(
      "faiss_store", 
      embeddings=embedding_model,
      allow_dangerous_deserialization=True  # explicitly allows loading .pkl safely
    )

# Print number of vectors
print(f"Number of vectors in FAISS DB: {len(db.index_to_docstore_id)}")

# Function to get size of folder
def get_folder_size(path):
    return sum(
        os.path.getsize(os.path.join(dirpath, filename))
        for dirpath, _, filenames in os.walk(path)
        for filename in filenames
    )

# Calculate and print DB size
size_bytes = get_folder_size("faiss_store")
size_mb = size_bytes / (1024 * 1024)
print(f"FAISS DB size on disk: {size_mb:.2f} MB")

In [ ]:
# View transaction_db
for transaction in transaction_db:
    print("Date/Time:", transaction['datetime'])
    print("Category:", transaction['category'])
    print("Input:", transaction['input'])
    print("Amount:", transaction['amount'])
    print("-" * 30)  # Separator for readability

In [ ]:
# # Delete locally stored vector db
# import shutil

# shutil.rmtree("faiss_store")
# print("Local FAISS vector DB deleted.")

# transaction_db = []